Bonus Code for Chapter 5
Alternative Weight Loading from Hugging Face Model Hub Via safetensors

In [ ]:
# 从 importlib.metadata 导入 version 函数，用于查询已安装第三方库的版本号
from importlib.metadata import version

# 本notebook依赖的关键库：numpy（数值计算）、torch（PyTorch 张量与模型）、safetensors（HuggingFace 的安全权重文件格式）
pkgs = ["numpy", "torch", "safetensors"]
for p in pkgs:
    # 逐个打印每个库的版本号，便于确认当前环境是否满足运行本notebook的要求
    print(f"{p} version: {version(p)}")

In [ ]:
# 【bug 修复】原代码写的是 `from Build_an_LLM_from_Scratch.ch04 import GPTModel`，
# 但实际发布的可导入包名是 `llms_from_scratch`（见下方安装说明链接），
# 使用 `Build_an_LLM_from_Scratch` 作为包名会导致 ModuleNotFoundError，这里改为正确的包名。
from llms_from_scratch.ch04 import GPTModel  # 导入第4章实现的自定义 GPTModel 类
# For llms_from_scratch installation instructions, see:
# https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg

In [ ]:
# BASE_CONFIG：GPT-2 系列模型共用的基础超参数配置
BASE_CONFIG = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "drop_rate": 0.0,       # Dropout rate
    "qkv_bias": True        # Query-key-value bias  （HF 官方 GPT-2 的 QKV 线性层带偏置项，这里必须设为 True 才能对齐权重形状）
}

# model_configs：不同规模 GPT-2 模型各自独有的结构超参数（嵌入维度、Transformer 层数、注意力头数）
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}


# 选择要加载的模型规模，可选："gpt2-small (124M)" / "gpt2-medium (355M)" / "gpt2-large (774M)" / "gpt2-xl (1558M)"
CHOOSE_MODEL = "gpt2-small (124M)"
# 将所选模型的专属配置合并进 BASE_CONFIG，得到该模型完整的初始化参数
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

In [ ]:
import os
import requests
# safetensors 是 HuggingFace 推出的安全、快速的张量序列化格式（相比 pickle 更安全，加载时不会执行任意代码）
from safetensors.torch import load_file

# URL_DIR：把本notebook里易读的模型别名映射到 HuggingFace Hub 上 openai-community 组织下对应的仓库目录名
URL_DIR = {
  "gpt2-small (124M)": "gpt2",         # works ok
  "gpt2-medium (355M)": "gpt2-medium", # this file seems to have issues via `generate`
  "gpt2-large (774M)": "gpt2-large",   # works ok
  "gpt2-xl (1558M)": "gpt2-xl"         # works ok
}

# 拼接出该模型 safetensors 权重文件在 HuggingFace Hub 上的直链下载地址
url = f"https://huggingface.co/openai-community/{URL_DIR[CHOOSE_MODEL]}/resolve/main/model.safetensors"
# 本地保存的文件名按模型区分，避免不同规模的模型权重互相覆盖
output_file = f"model-{URL_DIR[CHOOSE_MODEL]}.safetensors"

# Download file
# 如果本地还没有下载过该权重文件，则通过 HTTP GET 请求下载并写入本地磁盘
if not os.path.exists(output_file):
    response = requests.get(url, timeout=30)
    response.raise_for_status()  # 若响应状态码表示失败（如 404），则抛出异常，避免静默地把错误内容当权重加载
    with open(output_file, "wb") as f:
        f.write(response.content)

# Load file
# 用 safetensors 提供的 load_file 读取权重文件，返回一个 {参数名: torch.Tensor} 的字典，
# 即 HuggingFace 官方 GPT-2 checkpoint 原始命名风格的 state_dict
state_dict = load_file(output_file)

In [ ]:
def assign(left, right):
    # 校验待赋值的两个张量形状是否一致；形状不匹配通常意味着权重映射关系写错了，直接报错比静默出错更安全
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    # 用右侧（来自 HF safetensors 的权重）构造一个新的 nn.Parameter 返回；
    # detach() 用于脱离原来的计算图，避免意外的梯度追踪
    return torch.nn.Parameter(right.detach())
def load_weights_into_gpt(gpt, params):
    # params 是从 safetensors 加载出的 HF 原始命名风格的 state_dict，
    # 键名沿用了 HuggingFace/OpenAI 官方 GPT-2 checkpoint 的命名规则（如 "wte.weight"、"h.0.attn.c_attn.weight" 等），
    # 下面逐一把这些键映射到我们自己实现的 GPTModel 对应子模块上
    # wpe = word position embedding（位置嵌入），wte = word token embedding（词元嵌入）
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params["wpe.weight"])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params["wte.weight"])

    for b in range(len(gpt.trf_blocks)):
        # HF GPT-2 把 Q、K、V 三个投影合并存成了一个大的 c_attn 权重（在最后一维拼接），
        # 这里用 torch.chunk 按最后一维切成三份，分别还原出 Q/K/V 各自的权重；
        # 注：torch.chunk 支持用 axis 作为 dim 的别名（兼容 NumPy 习惯用法），效果等同于 dim=-1，这不是 bug
        q_w, k_w, v_w = torch.chunk(
            params[f"h.{b}.attn.c_attn.weight"], 3, axis=-1)
        # HF 的 c_attn 是 Conv1D 层，其权重形状是 [in_features, out_features]，
        # 而我们模型里的 W_query/W_key/W_value 是标准 nn.Linear，权重形状是 [out_features, in_features]，
        # 因此这里需要转置 .T 才能让形状对齐
        gpt.trf_blocks[b].att.W_query.weight = assign(
            gpt.trf_blocks[b].att.W_query.weight, q_w.T)
        gpt.trf_blocks[b].att.W_key.weight = assign(
            gpt.trf_blocks[b].att.W_key.weight, k_w.T)
        gpt.trf_blocks[b].att.W_value.weight = assign(
            gpt.trf_blocks[b].att.W_value.weight, v_w.T)

        # 同样地，合并存储的 Q/K/V 偏置也按最后一维切成三份；偏置是一维向量，转置与否没有影响，因此不需要 .T
        q_b, k_b, v_b = torch.chunk(
            params[f"h.{b}.attn.c_attn.bias"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.bias = assign(
            gpt.trf_blocks[b].att.W_query.bias, q_b)
        gpt.trf_blocks[b].att.W_key.bias = assign(
            gpt.trf_blocks[b].att.W_key.bias, k_b)
        gpt.trf_blocks[b].att.W_value.bias = assign(
            gpt.trf_blocks[b].att.W_value.bias, v_b)

        # c_proj 是注意力模块的输出投影（对应我们模型里的 out_proj），同样以 Conv1D 形式存储，权重需要转置
        gpt.trf_blocks[b].att.out_proj.weight = assign(
            gpt.trf_blocks[b].att.out_proj.weight,
            params[f"h.{b}.attn.c_proj.weight"].T)
        gpt.trf_blocks[b].att.out_proj.bias = assign(
            gpt.trf_blocks[b].att.out_proj.bias,
            params[f"h.{b}.attn.c_proj.bias"])

        # mlp.c_fc 是前馈网络（FeedForward）第一层线性层（升维），HF 同样用 Conv1D 存储，权重需要转置
        gpt.trf_blocks[b].ff.layers[0].weight = assign(
            gpt.trf_blocks[b].ff.layers[0].weight,
            params[f"h.{b}.mlp.c_fc.weight"].T)
        gpt.trf_blocks[b].ff.layers[0].bias = assign(
            gpt.trf_blocks[b].ff.layers[0].bias,
            params[f"h.{b}.mlp.c_fc.bias"])
        # mlp.c_proj 是前馈网络第二层线性层（降维回 emb_dim），对应我们模型 ff.layers[2]（layers[1] 是 GELU 激活函数）
        gpt.trf_blocks[b].ff.layers[2].weight = assign(
            gpt.trf_blocks[b].ff.layers[2].weight,
            params[f"h.{b}.mlp.c_proj.weight"].T)
        gpt.trf_blocks[b].ff.layers[2].bias = assign(
            gpt.trf_blocks[b].ff.layers[2].bias,
            params[f"h.{b}.mlp.c_proj.bias"])

        # ln_1 / ln_2 分别是每个 Transformer 块里的第一个和第二个 LayerNorm，
        # 对应我们模型里的 norm1 / norm2；HF 的 weight/bias 对应我们模型里的 scale/shift
        gpt.trf_blocks[b].norm1.scale = assign(
            gpt.trf_blocks[b].norm1.scale,
            params[f"h.{b}.ln_1.weight"])
        gpt.trf_blocks[b].norm1.shift = assign(
            gpt.trf_blocks[b].norm1.shift,
            params[f"h.{b}.ln_1.bias"])
        gpt.trf_blocks[b].norm2.scale = assign(
            gpt.trf_blocks[b].norm2.scale,
            params[f"h.{b}.ln_2.weight"])
        gpt.trf_blocks[b].norm2.shift = assign(
            gpt.trf_blocks[b].norm2.shift,
            params[f"h.{b}.ln_2.bias"])

    # ln_f 是最终输出前的 LayerNorm，对应我们模型里的 final_norm
    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["ln_f.weight"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["ln_f.bias"])
    # GPT-2 采用了权重绑定（weight tying）：输出层 out_head 与词嵌入 wte 共享同一份权重矩阵，
    # 因此这里再次复用 "wte.weight" 来初始化 out_head，而不是去查找一个单独存在的输出层权重
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte.weight"])

In [ ]:
import torch


# 用前面合并好的 BASE_CONFIG 实例化我们自己实现的 GPTModel（此时权重是随机初始化的）
gpt = GPTModel(BASE_CONFIG)

# 优先使用 GPU（cuda），否则退回 CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# 把从 HF safetensors 加载并完成键名映射的权重灌入模型（原地修改 gpt 各层的参数）
load_weights_into_gpt(gpt, state_dict)
# 将模型迁移到目标设备；末尾分号用于抑制 Jupyter 对该表达式返回值的自动输出
gpt.to(device);

In [ ]:
import tiktoken
# 【bug 修复】同前面的 cell，原代码里的 `Build_an_LLM_from_Scratch` 不是真实的可导入包名，
# 应改为实际发布的 `llms_from_scratch` 包，否则会因 ModuleNotFoundError 而无法运行
from llms_from_scratch.ch05 import generate, text_to_token_ids, token_ids_to_text


# 固定随机种子，保证多次运行的生成结果可复现
torch.manual_seed(123)

# 使用 GPT-2 官方的 BPE 分词器（tiktoken 提供的 "gpt2" 编码）
tokenizer = tiktoken.get_encoding("gpt2")

# 用加载好权重的模型进行自回归文本生成
token_ids = generate(
    model=gpt.to(device),
    idx=text_to_token_ids("Every effort moves", tokenizer).to(device),  # 把起始提示文本编码为 token id 序列
    max_new_tokens=30,   # 最多生成 30 个新 token
    context_size=BASE_CONFIG["context_length"],  # 生成时使用的上下文窗口大小
    top_k=1,             # top_k=1 等价于贪心解码，每一步只取概率最高的 token
    temperature=1.0
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))